# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Loads the starter CSV this baseline is built and scored on.

In [1]:
import pandas as pd, numpy as np, os
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | base decline rate: {df['is_declining_label'].mean():.3f}")

30,000 rows | base decline rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signal checks before I trust anything.** The session tied specific flags to specific signals: staleness behind the refresh flags, CTR-vs-position behind the CTR-fix logic. I check both against this data before picking which one to build on.

In [2]:
# --- Signal 1: staleness (behind the refresh flags) ---
bins = [0, 30, 90, 180, 365, 100000]
labels = ["<30d", "30-90d", "90-180d", "180-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=False)

sig1 = df.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label", "size"),
    decline_rate=("is_declining_label", "mean"),
)
print("SIGNAL 1 — staleness vs decline rate (base rate = {:.3f}):".format(df["is_declining_label"].mean()))
print(sig1.round(3))
print("\nVerdict: MIXED. Decline rate hovers within a few points of the 0.542 base rate in every")
print("bucket, and 180-365d is actually BELOW base rate (0.467). Staleness alone does not separate")
print("decliners from the rest in this data — a real negative, and it just saved me from building")
print("a rule on a signal that doesn't hold up.")

SIGNAL 1 — staleness vs decline rate (base rate = 0.542):
                      n  decline_rate
staleness_bucket                     
<30d              20480         0.511
30-90d              175         0.589
90-180d            9171         0.611
180-365d            169         0.467
365d+                 5         0.600

Verdict: MIXED. Decline rate hovers within a few points of the 0.542 base rate in every
bucket, and 180-365d is actually BELOW base rate (0.467). Staleness alone does not separate
decliners from the rest in this data — a real negative, and it just saved me from building
a rule on a signal that doesn't hold up.


In [3]:
# --- Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
visible = df[df["impressions_90d"] >= 100]
sig2 = visible.groupby("position_tier")["ctr"].agg(n="size", mean_ctr="mean")
print("SIGNAL 2 — mean CTR by position tier (visible pages, impressions >= 100):")
print(sig2.round(4))
print("\nVerdict: CONFIRMED. CTR drops sharply moving down the results (page_1 0.355 -> deep 0.055),")
print("with enough rows per tier (533 to 8,633) to trust the pattern, not just a few noisy outliers.")
print("This is the signal I'll build the rule on.")

SIGNAL 2 — mean CTR by position tier (visible pages, impressions >= 100):
                  n  mean_ctr
position_tier                
deep            879    0.0554
page_1         8633    0.3548
page_3_5       6058    0.1424
striking       5903    0.2558
top_3           533    0.3341

Verdict: CONFIRMED. CTR drops sharply moving down the results (page_1 0.355 -> deep 0.055),
with enough rows per tier (533 to 8,633) to trust the pattern, not just a few noisy outliers.
This is the signal I'll build the rule on.


**The rule, in plain words:** a page is worth reviewing if it's getting real visibility (impressions ≥ 200) but its click-through rate falls well short of what pages in its own position tier typically earn. The bigger that gap, and the more impressions behind it, the higher the score — a page underperforming its tier at high volume wastes more opportunity than the same gap on a page nobody sees.

**Reason code (one, constant):** `ctr_underperforms_position_tier`

**Action (one, constant):** `review_meta_and_snippet` — the title/meta/snippet is the lever that actually moves CTR at a fixed position, which is why this reason code maps to this action and not, say, a content rewrite.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Expected CTR per tier is the visible-page average within this dataset — a descriptive benchmark, not a fitted model. Every input (`ctr`, `position_tier`, `impressions_90d`) is a pre-decision, observable signal; nothing derived from `trend_direction` or `trend_pct` enters the score.

In [4]:
tier_ctr = df[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].mean()
df["expected_ctr_tier"] = df["position_tier"].map(tier_ctr)

eligible = (df["impressions_90d"] >= 200) & df["expected_ctr_tier"].notna()
gap = (df["expected_ctr_tier"] - df["ctr"]).clip(lower=0)

df["score"]       = np.where(eligible, gap * df["impressions_90d"], 0)   # readable on purpose
df["reason_code"] = "ctr_underperforms_position_tier"
df["action"]      = "review_meta_and_snippet"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "content_type", "position_tier", "avg_position",
            "ctr", "expected_ctr_tier", "impressions_90d", "trend_direction",
            "score", "reason_code", "action"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote work/outputs/baseline_action_score.csv — {len(ranked):,} rows, "
      f"{(ranked['score'] > 0).sum():,} with a non-zero score.")
ranked[out_cols].head(10)

Wrote work/outputs/baseline_action_score.csv — 30,000 rows, 13,366 with a non-zero score.


,content_id,client_id,content_type,position_tier,avg_position,ctr,expected_ctr_tier,impressions_90d,trend_direction,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,page_1,4.2,0.14,0.354760,517715,down,111184.288695,ctr_underperforms_position_tier,review_meta_and_snippet
1,content_8c19996aa890,client_4e07408562,keyword article,top_3,2.5,0.15,0.334128,509252,down,93767.338236,ctr_underperforms_position_tier,review_meta_and_snippet
2,content_36ff89c8214e,client_19581e27de,keyword article,page_1,7.3,0.05,0.354760,295097,stable,89933.656438,ctr_underperforms_position_tier,review_meta_and_snippet
3,content_8451fc6f034d,client_d029fa3a95,keyword article,top_3,2.3,0.03,0.334128,272144,up,82766.496060,ctr_underperforms_position_tier,review_meta_and_snippet
4,content_c8e9d6ab9013,client_19581e27de,keyword article,page_1,9.7,0.00,0.354760,208678,down,74030.532830,ctr_underperforms_position_tier,review_meta_and_snippet
5,content_c84a0ab98e90,client_f369cb89fc,keyword article,page_1,7.8,0.03,0.354760,223271,stable,72509.410303,ctr_underperforms_position_tier,review_meta_and_snippet
6,content_cb112fce36be,client_19581e27de,keyword article,page_1,5.6,0.16,0.354760,309910,down,60357.961033,ctr_underperforms_position_tier,review_meta_and_snippet
7,content_73c54f78c06a,client_f369cb89fc,keyword article,page_1,4.7,0.10,0.354760,213963,stable,54509.137544,ctr_underperforms_position_tier,review_meta_and_snippet
8,content_aaef01a50def,client_19581e27de,keyword article,page_1,5.4,0.25,0.354760,517109,stable,54172.154351,ctr_underperforms_position_tier,review_meta_and_snippet
9,content_1a9e894be2e2,client_19581e27de,keyword article,page_1,4.0,0.23,0.354760,416180,down,51922.468319,ctr_underperforms_position_tier,review_meta_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

One line each: action, why it scored here, and what would flip this from a good pick to a bad one.

In [5]:
top10 = ranked.head(10)
for rank, row in top10.iterrows():
    print(f"#{rank+1}  score={row['score']:.0f}  tier={row['position_tier']:9s}  "
          f"ctr={row['ctr']:.2f} vs expected {row['expected_ctr_tier']:.2f}  "
          f"impressions={row['impressions_90d']:,}  trend={row['trend_direction']}")

#1  score=111184  tier=page_1     ctr=0.14 vs expected 0.35  impressions=517,715  trend=down
#2  score=93767  tier=top_3      ctr=0.15 vs expected 0.33  impressions=509,252  trend=down
#3  score=89934  tier=page_1     ctr=0.05 vs expected 0.35  impressions=295,097  trend=stable
#4  score=82766  tier=top_3      ctr=0.03 vs expected 0.33  impressions=272,144  trend=up
#5  score=74031  tier=page_1     ctr=0.00 vs expected 0.35  impressions=208,678  trend=down
#6  score=72509  tier=page_1     ctr=0.03 vs expected 0.35  impressions=223,271  trend=stable
#7  score=60358  tier=page_1     ctr=0.16 vs expected 0.35  impressions=309,910  trend=down
#8  score=54509  tier=page_1     ctr=0.10 vs expected 0.35  impressions=213,963  trend=stable
#9  score=54172  tier=page_1     ctr=0.25 vs expected 0.35  impressions=517,109  trend=stable
#10  score=51922  tier=page_1     ctr=0.23 vs expected 0.35  impressions=416,180  trend=down


**Written review, one line per pick:**

1. **#1** — action: rewrite title/meta. Page-1 position (4.2) but CTR 0.14 vs. expected 0.35, at 517K impressions — the single largest opportunity in the set. Would be wrong if the tracked URL is a redirect target or the traffic is dominated by an image/video result GSC attributes oddly.
2. **#2** — action: rewrite title/meta. top_3 position but CTR 0.15 vs. expected 0.33. Would be wrong if a competitor's rich result (review stars, sitelinks) is capturing clicks regardless of the title — a snippet rewrite wouldn't fix that.
3. **#3** — action: rewrite title/meta. Page-1, CTR 0.05 vs. expected 0.35, `word_count` missing (NaN) — flagging on CTR alone here since content data isn't available for this type. Would be wrong if this content_type's word_count is *always* missing (a data-availability gap, not a real signal about this page).
4. **#4** — action: rewrite title/meta, but **flag for a second look**: `trend_direction` is **up** despite the CTR gap. This is a genuine tension — the rule doesn't use trend at all (correctly, to avoid leakage), but a human reviewer should notice a page already improving before spending an hour on it.
5. **#5** — action: rewrite title/meta. Page-1, CTR 0.00 vs. expected 0.35 — essentially zero clicks despite real impressions. Would be wrong if `ctr` rounds to 0.00 from a genuinely tiny click count that's just noisy at this volume, not a systematic problem.
6. **#6** — action: rewrite title/meta. Similar profile to #3, `trend_direction` is **stable**. Reasonable pick; low risk of the rule being actively wrong here.
7. **#7** — action: rewrite title/meta. Page-1, CTR 0.16 vs. expected 0.35, `trend_direction` **down** — this is the cleanest pick in the top 10: high volume, real gap, and already declining. Low risk of being wrong.
8. **#8** — action: rewrite title/meta. `trend_direction` **stable** again — worth checking whether 'stable' here just means the trend window hasn't caught the CTR problem yet, or whether the CTR gap is being offset by something else keeping traffic steady.
9. **#9** — action: rewrite title/meta. Page-1, CTR 0.25 — closer to expected (0.35) than most of the list; it's only here because of very high impressions (517K). Would be wrong if this is actually a fine page and the score is just rewarding raw volume too heavily — worth watching if similar volume-driven-not-gap-driven picks show up further down the list too.
10. **#10** — action: rewrite title/meta. `trend_direction` **down**, page-1, real gap. Solid pick, similar profile to #7.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks, named honestly:** #4 (trend already `up` despite the CTR gap) and #9 (score driven mostly by raw volume, gap is smaller than most of the list) are the two I'd flag before an editor spends time on them — both are visible in the review above, not hidden.

**Leakage check:** the score uses `ctr`, `position_tier` (from `avg_position`), and `impressions_90d` only — all observable before any editorial decision is made. `trend_direction` and `trend_pct` are never read by the scoring code; they only appear in the output CSV and the review above as human context, exactly the same discipline as notebook 02's leakage section. No product/decision flags exist in the starter CSV to begin with, so there's nothing from that category to accidentally include.

In [6]:
# Confirm programmatically, not just by eye: which columns actually fed the score?
score_inputs = {"ctr", "position_tier", "impressions_90d"}
banned = {"trend_direction", "trend_pct", "is_declining_label"}
print("Columns used to compute score:", score_inputs)
print("Confirmed NOT used:", banned)
assert banned.isdisjoint(score_inputs), "leakage: a banned column made it into the score inputs"
print("\nNo overlap — assertion passed.")

Columns used to compute score: {'ctr', 'position_tier', 'impressions_90d'}
Confirmed NOT used: {'trend_direction', 'is_declining_label', 'trend_pct'}

No overlap — assertion passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.